# Day 3 — 异常值检测：3σ 准则 vs IQR 四分位距法

> **日期**：2026-07-05  
> **主题**：两大主流量化方法的原理、对比与选型指南

## 核心概念

**异常值（Outlier）**：显著偏离整体分布的极端值，会严重干扰均值、方差、回归系数，建模前必须识别。

两种方法的本质区别：

| | 3σ | IQR |
|:---|:---|:---|
| **依赖** | 均值和标准差 | 分位数（Q1, Q3） |
| **适用分布** | 仅正态 | 任意分布 |
| **抗异常污染** | ❌ 差 | ✅ 强 |
| **可视化** | 无 | 箱线图原生配套 |

---

## 一、3σ 准则（3 倍标准差法）

### 理论基础

假设数据 ~ N(μ, σ²)：

| 区间 | 覆盖率 |
|:---|:---|
| μ ± σ | 68.27% |
| μ ± 2σ | 95.45% |
| **μ ± 3σ** | **99.73%** |

> 仅 0.27% 落在 ±3σ 外 ← 判为异常。

### 判定公式

```
Lower = μ − 3σ
Upper = μ + 3σ
```

x < Lower 或 x > Upper → **异常**。

### ✅ 优点
- 计算极简，一行代码
- 正态下有严格概率保证（99.73%）
- 工业传感器、时序数据首选

### ❌ 致命缺陷
- **被异常值反噬**：一个极端值即可拉偏 μ 和 σ，边界被「污染」，漏判真正的异常
- **分布依赖**：偏态、长尾数据完全失效
- 小样本下不可靠

> 💡 **一句话**：3σ 用「被污染的尺」去量异常——尺不准了，量出来的结果自然不可信。

---

## 二、IQR 四分位距法

### 基础定义

```
Q₁ = P₂₅（第 25 百分位）
Q₂ = P₅₀（中位数）
Q₃ = P₇₅（第 75 百分位）
IQR = Q₃ − Q₁（中间 50% 数据的跨度）
```

### 判定边界（1.5×IQR 标准）

```
Lower = Q₁ − 1.5 × IQR
Upper = Q₃ + 1.5 × IQR
```

> 严苛模式：3×IQR，仅抓取极端离群。

### ✅ 核心优势

| 优势 | 原因 |
|:---|:---|
| **鲁棒抗污染** | 分位数不看极端值，一个 outlier 不影响 Q1/Q3 |
| **无分布要求** | 正态、偏态、长尾、离散全适用 |
| **箱线图原生** | Boxplot 的须 = 1.5×IQR，可视化直观 |
| **小样本可用** | 远比 3σ 稳定 |

### ❌ 缺点
- 纯经验规则，无概率理论支撑
- 理想正态下灵敏度略低于 3σ

> 💡 **一句话**：IQR 用「中位数的尺」——极端值再多也污染不了分位数，找出来的异常才是真异常。

---

## 三、核心对比

| 维度 | 3σ | IQR |
|:---|:---|:---|
| 理论基础 | 正态概率分布 | 分位数经验规则 |
| 抗异常污染 | ❌ 差（μ 被拉偏） | ✅ 强（分位数免疫） |
| 分布要求 | 仅正态 | **任意** |
| 核心指标 | μ, σ | Q₁, Q₃, IQR |
| 可视化 | 无 | 箱线图直接对应 |
| 小样本 | 差 | 稳定 |
| 典型场景 | 传感器、实验数据 | 收入、销量、用户行为 |

### 分布形态决定选择

```
正态钟形   → 3σ / IQR 均可，结果高度重合
右偏长尾   → 必须 IQR（3σ 边界被拉宽，漏判大量异常）
存在极端值 → 必须 IQR（3σ 的 μ 和 σ 已被污染）
```

---

## 四、选型速查

| 场景 | 推荐 | 理由 |
|:---|:---:|:---|
| 传感器、工业时序 | 3σ | 近似正态、计算快 |
| 收入、销售额、订单金额 | **IQR** | 典型右偏长尾 |
| 用户时长、点击量 | **IQR** | 极偏分布 |
| 学术实验测量数据 | 3σ | 正态假设成立 |
| 日常 EDA、箱线图 | **IQR** | 原生配套、鲁棒 |
| 不确定分布形态 | **IQR** | 不依赖假设，保险之选 |

> 🏆 **黄金法则**：拿不准分布 → 用 IQR。正态下 IQR 不会错，偏态下 3σ 一定错。

---

## 五、Python 速查

### 3σ

```python
mu, sigma = data.mean(), data.std(ddof=1)
outliers = data[(data < mu - 3*sigma) | (data > mu + 3*sigma)]
```

### IQR

```python
q1, q3 = np.percentile(data, [25, 75])
iqr = q3 - q1
outliers = data[(data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)]
```

### DataFrame 批量检测

```python
def detect_outliers_iqr(df, col, k=1.5):
    """返回布尔掩码：True = 异常"""
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    return (df[col] < q1 - k*iqr) | (df[col] > q3 + k*iqr)
```

| k 值 | 含义 |
|:---|:---|
| 1.5 | 常规异常（默认） |
| 3.0 | 极端异常（严苛） |

---

## 六、核心结论

1. **3σ = 相信数据是正态的**，用均值衡量一切——但均值会被异常值拖垮
2. **IQR = 不信任任何极端值**，用中位数和分位数——极端值再多也动不了分位数
3. 日常 EDA **默认用 IQR**，只有确认数据近似正态时才考虑 3σ
4. 两者在正态数据下结果高度重合，**分歧越大说明数据越偏态**

---

# Part B — 匈牙利算法 vs Hopcroft–Karp 算法

> 二分图最大匹配的两种经典解法，从 O(nE) 到 O(√n·E) 的跨越。

## 一、核心区别

| | 匈牙利（朴素 DFS） | Hopcroft–Karp |
|:---|:---|:---|
| **策略** | 单点 DFS，找一条就停 | BFS 分层 + 批量 DFS |
| **每轮增益** | ≤1 组匹配 | 多条不相交增广路同时更新 |
| **剪枝** | 无 | dx 数组锁死最短路径 |
| **复杂度** | O(n·E) | O(√n·E) |
| **适用** | 小图、教学理解 | 大规模二分图 |

> 匈牙利 = 暴力单挑； HK = 先勘探（BFS）再批量收割（DFS）。

## 二、BFS 分层与 dx 数组

### dx 数组的作用

BFS 从所有自由左点出发，计算到达每个左节点的**最短交替路径层数**。DFS 只允许走  的边，过滤所有绕路。

### BFS 阶段行为

| 行为 | 说明 |
|:---|:---|
| 不修改匹配 | 纯勘探，只填 dx |
| 不全终止 | 即使遇到空岗也不停，必须算完所有层 |
| has_aug | 只要有任意分支抵达空岗即为 true |

### 分层路径特性

- **同一自由点出发**的所有合法增广路，**路径长度必然相等**——更长路径中途被 dx 截断
- 不同自由点的最短增广路长度**可以不同**，只有等于全局最短的才被本轮使用
- BFS 分叉出的多条备选路径层数一致，DFS 只取邻接表靠前的一条，其余剪枝

## 三、DFS 阶段执行逻辑

### 通行约束



### 匹配更新

- 找到增广路 → **立刻翻转**匹配边，向上回溯逐层更新
- 同起点多条备选分支 → **只用第一条**，找到即停
- 不同自由点的增广路 → 串行遍历，但分层图保证**顶点互不相交**，更新不会冲突

> DFS 不是多线程并行，但分层图的隔离性让批量更新等价于并行效果。

## 四、增广路关键性质

| 性质 | 匈牙利 | HK |
|:---|:---|:---|
| **起点** | 必须是自由左点（两者相同） | 必须是自由左点（两者相同） |
| **每轮条数** | 1 条 | 多条（不相交） |
| **路径相交** | 可能重复搜同一条 | 分层图内最短增广路**完全不相交** |
| **选择逻辑** | 邻接表顺序决定 | BFS 筛最短 + 邻接表顺序裁决 |

> 只有被 dx 过滤掉的**长路径**才会相交——DFS 根本不会遍历它们。

## 五、流程对比



| 对比 | 匈牙利 | HK |
|:---|:---|:---|
| 是否区分路径长短 | ❌ | ✅ 始终最短优先 |
| 是否批量 | ❌ | ✅ 一轮多条 |
| 无效搜索 | 多（无分层剪枝） | 少（dx 大量剪枝） |

## 六、一句话总结

> **匈牙利** = 暴力单点 DFS，找到一条是一条，简单但 O(nE)。
> **HK** = BFS 全局勘探 + DFS 批量收割，靠 dx 分层剪枝砍到 O(√n·E)，大规模二分图首选。